# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

> **GPU recommended.** Go to **Runtime → Change runtime type → T4 GPU** before running. LSTM training is significantly faster on GPU.

---

# Sentiment Analysis Case Study

Apply NLP preprocessing from module 15 and deep learning architectures from module 14 to a real sentiment classification pipeline — from raw text to a trained and evaluated deep learning model.

This notebook covers all three lessons in module 16:
1. Problem Definition & Data Collection
2. NLP Preprocessing & EDA
3. Model Building & Evaluation

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score
)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---

# Part 1: Problem Definition & Data Collection

## What Is Sentiment Analysis?

Sentiment analysis is binary (or multi-class) text classification: given a piece of text, predict whether it expresses a positive, negative, or neutral opinion. It is one of the most widely deployed NLP applications in industry:

| Use case | Input text | Output |
|----------|-----------|--------|
| Product review monitoring | "The battery dies after 2 hours" | Negative |
| Social media triage | "Absolutely love this update!" | Positive |
| Customer support routing | "This is the third time I've called" | Negative |
| Brand tracking | "Stock just hit an all-time high" | Positive |

Sentiment is harder than it looks. Negation, sarcasm, and domain-specific language all cause failures in simple models:
- *"The movie was not bad"* — negation makes this positive
- *"Oh great, another delay"* — sarcasm makes this negative
- *"The gore was surprisingly well done"* — domain context (horror film) changes interpretation

## The IMDB Dataset

The IMDB Movie Reviews dataset contains 50,000 movie reviews labeled as positive (1) or negative (0). It is split evenly into 25,000 training and 25,000 test examples, with exactly 50% positive and 50% negative in each split.

Keras provides a pre-processed version where each review is already converted to a sequence of integers — each integer is the rank of the corresponding word by frequency in the corpus. For example, integer 4 maps to the 4th most common word.

In [ ]:
VOCAB_SIZE = 10000   # keep only the 10,000 most frequent words

(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f"Training samples : {len(X_train_raw):,}")
print(f"Test samples     : {len(X_test_raw):,}")
print(f"\nClass distribution (train):")
print(f"  Positive : {y_train.sum():,} ({y_train.mean():.1%})")
print(f"  Negative : {(1 - y_train).sum():,} ({(1 - y_train).mean():.1%})")

## Decoding Reviews

The raw sequences are integers. To read them as text, we build a reverse lookup from integer → word using the word index Keras provides.

In [ ]:
# Keras reserves the first 3 indices for special tokens:
# 0=<PAD>, 1=<START>, 2=<UNK>
# The word index is offset by 3 to accommodate these
word_index = imdb.get_word_index()
reverse_index = {v + 3: k for k, v in word_index.items()}
reverse_index[0] = "<PAD>"
reverse_index[1] = "<START>"
reverse_index[2] = "<UNK>"

def decode_review(sequence, n_words=None):
    """Convert integer sequence back to readable text, skipping padding."""
    tokens = [reverse_index.get(i, "?") for i in sequence if i != 0]
    if n_words:
        tokens = tokens[:n_words]
    return " ".join(tokens)

In [ ]:
# Print 4 sample reviews (2 positive, 2 negative)
pos_indices = np.where(y_train == 1)[0][:2]
neg_indices = np.where(y_train == 0)[0][:2]

for idx, label_name in [(pos_indices[0], "POSITIVE"), (pos_indices[1], "POSITIVE"),
                         (neg_indices[0], "NEGATIVE"), (neg_indices[1], "NEGATIVE")]:
    print(f"[{label_name}] (length: {len(X_train_raw[idx])} tokens)")
    print(decode_review(X_train_raw[idx], n_words=60))
    print()

## Baseline: Majority Class

Before building any model, establish the floor: a classifier that always predicts the most common class.

In [ ]:
majority_class = int(y_train.mean() >= 0.5)
baseline_acc   = max(y_train.mean(), 1 - y_train.mean())

print(f"Majority class   : {'Positive' if majority_class == 1 else 'Negative'}")
print(f"Baseline accuracy: {baseline_acc:.1%}")
print()
print("Any model we build must beat 50.0% to be better than random guessing.")

---

# Part 2: NLP Preprocessing & EDA

## Review Length Distribution

Sequence models require all inputs to be the same length. Understanding the length distribution tells us how much content we will discard (truncation) and how much padding we will add.

In [ ]:
train_lengths = np.array([len(seq) for seq in X_train_raw])

print("Review length statistics (train):")
print(f"  Mean   : {train_lengths.mean():.0f} tokens")
print(f"  Median : {np.median(train_lengths):.0f} tokens")
print(f"  Min    : {train_lengths.min()} tokens")
print(f"  Max    : {train_lengths.max()} tokens")
print(f"  p90    : {np.percentile(train_lengths, 90):.0f} tokens")
print(f"  p95    : {np.percentile(train_lengths, 95):.0f} tokens")
print()
for max_len in [100, 200, 300, 500]:
    pct = np.mean(train_lengths <= max_len)
    print(f"  Reviews fully covered by MAX_LEN={max_len}: {pct:.1%}")

In [ ]:
pos_lengths = train_lengths[y_train == 1]
neg_lengths = train_lengths[y_train == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall distribution
axes[0].hist(train_lengths, bins=60, color="steelblue", edgecolor="white")
axes[0].axvline(200, color="#e74c3c", linestyle="--", linewidth=1.5, label="MAX_LEN=200")
axes[0].axvline(np.median(train_lengths), color="orange", linestyle="--",
                linewidth=1.5, label=f"Median={int(np.median(train_lengths))}")
axes[0].set_title("Review Length Distribution")
axes[0].set_xlabel("Tokens")
axes[0].set_ylabel("Count")
axes[0].legend()

# By sentiment
axes[1].hist(pos_lengths, bins=60, alpha=0.6, color="#2ecc71", label="Positive")
axes[1].hist(neg_lengths, bins=60, alpha=0.6, color="#e74c3c", label="Negative")
axes[1].axvline(200, color="black", linestyle="--", linewidth=1.2, label="MAX_LEN=200")
axes[1].set_title("Review Length by Sentiment")
axes[1].set_xlabel("Tokens")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

## Vocabulary Analysis: Top Words by Sentiment

Which words appear most in positive vs. negative reviews? Some will be generic ("the", "and") — these are stopwords that appear in both classes equally. The interesting words are those that appear far more often in one class.

In [ ]:
# Collect all word tokens, split by sentiment
# Skip special tokens (indices 0, 1, 2) — keep only real words
pos_tokens = [token for i, seq in enumerate(X_train_raw)
              if y_train[i] == 1
              for token in seq if token > 2]

neg_tokens = [token for i, seq in enumerate(X_train_raw)
              if y_train[i] == 0
              for token in seq if token > 2]

pos_word_counts = Counter(reverse_index.get(t, "?") for t in pos_tokens)
neg_word_counts = Counter(reverse_index.get(t, "?") for t in neg_tokens)

print("Top 20 words in POSITIVE reviews:")
print([w for w, _ in pos_word_counts.most_common(20)])

print("\nTop 20 words in NEGATIVE reviews:")
print([w for w, _ in neg_word_counts.most_common(20)])

In [ ]:
# Compute relative frequency: how much more often does a word appear in one class?
# Use only words that appear >= 500 times in total to focus on reliable signal
all_words  = set(pos_word_counts.keys()) & set(neg_word_counts.keys())
total_pos  = sum(pos_word_counts.values())
total_neg  = sum(neg_word_counts.values())

ratios = {}
for w in all_words:
    if pos_word_counts[w] + neg_word_counts[w] < 500:
        continue
    pos_rate = pos_word_counts[w] / total_pos
    neg_rate = neg_word_counts[w] / total_neg
    if neg_rate > 0:
        ratios[w] = pos_rate / neg_rate

ratio_series = pd.Series(ratios).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Most negative-skewed words
neg_skewed = ratio_series.head(15)
axes[0].barh(neg_skewed.index, neg_skewed.values, color="#e74c3c")
axes[0].set_title("Words skewed toward NEGATIVE reviews")
axes[0].set_xlabel("Positive rate / Negative rate")

# Most positive-skewed words
pos_skewed = ratio_series.tail(15).sort_values(ascending=True)
axes[1].barh(pos_skewed.index, pos_skewed.values, color="#2ecc71")
axes[1].set_title("Words skewed toward POSITIVE reviews")
axes[1].set_xlabel("Positive rate / Negative rate")

plt.suptitle("Word Frequency Ratio: Positive vs Negative", fontsize=13)
plt.tight_layout()
plt.show()

## Sequence Padding

All models require fixed-length inputs. We set `MAX_LEN=200`, which covers ~75% of reviews fully. Reviews shorter than 200 tokens are padded with zeros at the end (`padding='post'`); longer reviews are trimmed from the end (`truncating='post'`).

**Trade-off:** A larger `MAX_LEN` preserves more content but increases memory and compute. MAX_LEN=200 is a good balance for this dataset.

In [ ]:
MAX_LEN = 200

X_train = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding="post", truncating="post")
X_test  = pad_sequences(X_test_raw,  maxlen=MAX_LEN, padding="post", truncating="post")

print(f"After padding:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  dtype   : {X_train.dtype}")
print()
print(f"Reviews fully preserved (length <= {MAX_LEN}): "
      f"{np.mean(train_lengths <= MAX_LEN):.1%}")
print(f"Reviews truncated: "
      f"{np.mean(train_lengths > MAX_LEN):.1%}")

---

# Part 3: Model Building & Evaluation

We build three models in increasing order of complexity, then compare them:

| Model | Architecture | Key idea |
|-------|-------------|----------|
| **Baseline** | BoW + Logistic Regression | Treats each word independently; no order |
| **LSTM** | Embedding + LSTM | Reads sequence left-to-right; captures order |
| **BiLSTM** | Embedding + Bidirectional LSTM + Dropout | Reads both directions; less overfitting |

## Helper: Plot Training Curves

In [ ]:
def plot_history(history, title):
    """Plot loss and accuracy curves for a Keras training history."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(history.history["loss"],     label="Train", marker="o", ms=4)
    axes[0].plot(history.history["val_loss"], label="Val",   marker="s", ms=4)
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Binary cross-entropy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history["accuracy"],     label="Train", marker="o", ms=4)
    axes[1].plot(history.history["val_accuracy"], label="Val",   marker="s", ms=4)
    axes[1].set_title(f"{title} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Model 1: Bag-of-Words + Logistic Regression (Baseline)

Before any deep learning, establish how well a simple model performs. Convert each review to a binary feature vector: entry `j` = 1 if word index `j` appears in the review, 0 otherwise.

In [ ]:
def sequences_to_bow(sequences, vocab_size):
    """Convert integer sequences to a binary bag-of-words matrix."""
    matrix = np.zeros((len(sequences), vocab_size), dtype="float32")
    for i, seq in enumerate(sequences):
        for j in seq:
            if j > 0:   # skip padding token
                matrix[i, j] = 1.0
    return matrix

print("Building BoW matrices (may take ~30 seconds)...")
X_bow_train = sequences_to_bow(X_train, VOCAB_SIZE)
X_bow_test  = sequences_to_bow(X_test,  VOCAB_SIZE)
print(f"BoW matrix shape: {X_bow_train.shape}")

In [ ]:
lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_bow_train, y_train)

y_pred_lr   = lr_model.predict(X_bow_test)
y_proba_lr  = lr_model.predict_proba(X_bow_test)[:, 1]

print("=== Model 1: BoW + Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=["Negative", "Positive"]))
print(f"AUC: {roc_auc_score(y_test, y_proba_lr):.4f}")

## Model 2: Embedding + LSTM

The LSTM reads the review token-by-token, maintaining a hidden state that carries context forward through the sequence. Unlike the BoW model, it knows that "not boring" differs from "boring".

```
Embedding layer: word_index → 64-d vector (learned)
        ↓
LSTM:   h₁ → h₂ → h₃ → ... → hₙ  (hidden state carries context)
        ↓
Dense(1, sigmoid): binary prediction
```

In [ ]:
EMBED_DIM = 64

model_lstm = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    layers.LSTM(64, dropout=0.2),
    layers.Dense(1, activation="sigmoid"),
], name="lstm")

model_lstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_lstm.summary()

In [ ]:
history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
    verbose=1,
)

In [ ]:
plot_history(history_lstm, "LSTM")

In [ ]:
y_proba_lstm = model_lstm.predict(X_test, verbose=0).ravel()
y_pred_lstm  = (y_proba_lstm >= 0.5).astype(int)

print("=== Model 2: LSTM ===")
print(classification_report(y_test, y_pred_lstm, target_names=["Negative", "Positive"]))
print(f"AUC: {roc_auc_score(y_test, y_proba_lstm):.4f}")

## Model 3: Embedding + Bidirectional LSTM + Dropout

A `Bidirectional` wrapper runs two LSTMs in parallel — one reads left-to-right, the other right-to-left — and concatenates their outputs. This gives the model context from both directions at every position.

```
  Forward LSTM:   h₁→ h₂→ h₃→ ... →hₙ
                               ↓ concatenate
  Backward LSTM:  h₁← h₂← h₃← ... ←hₙ
        ↓
  Dropout(0.5)  →  Dense(1, sigmoid)
```

`Dropout(0.5)` randomly zeros half the units during training, forcing the model to learn redundant representations — this reduces overfitting.

In [ ]:
model_bilstm = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2)),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
], name="bidirectional_lstm")

model_bilstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_bilstm.summary()

In [ ]:
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
    verbose=1,
)

In [ ]:
plot_history(history_bilstm, "Bidirectional LSTM")

In [ ]:
y_proba_bilstm = model_bilstm.predict(X_test, verbose=0).ravel()
y_pred_bilstm  = (y_proba_bilstm >= 0.5).astype(int)

print("=== Model 3: Bidirectional LSTM + Dropout ===")
print(classification_report(y_test, y_pred_bilstm, target_names=["Negative", "Positive"]))
print(f"AUC: {roc_auc_score(y_test, y_proba_bilstm):.4f}")

## Model Comparison

In [ ]:
rows = []
for name, pred, proba in [
    ("BoW + Logistic Regression", y_pred_lr,     y_proba_lr),
    ("LSTM",                       y_pred_lstm,   y_proba_lstm),
    ("Bidirectional LSTM",         y_pred_bilstm, y_proba_bilstm),
]:
    rows.append({
        "Model":     name,
        "Accuracy":  f"{(pred == y_test).mean():.4f}",
        "Precision": f"{precision_score(y_test, pred):.4f}",
        "Recall":    f"{recall_score(y_test, pred):.4f}",
        "F1":        f"{f1_score(y_test, pred):.4f}",
        "AUC":       f"{roc_auc_score(y_test, proba):.4f}",
    })

print(pd.DataFrame(rows).to_string(index=False))

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name, proba, color in [
    ("BoW + LR",    y_proba_lr,     "#3498db"),
    ("LSTM",        y_proba_lstm,   "#e67e22"),
    ("BiLSTM",      y_proba_bilstm, "#2ecc71"),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name}  (AUC={auc:.3f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Confusion Matrix: Best Model

In [ ]:
# Use the bidirectional LSTM as the best model
cm = confusion_matrix(y_test, y_pred_bilstm)
disp = ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Bidirectional LSTM")
plt.show()

print(f"Correct predictions  : {cm.diagonal().sum():,} / {cm.sum():,}")
print(f"False positives      : {cm[0, 1]:,}  (negative review predicted positive)")
print(f"False negatives      : {cm[1, 0]:,}  (positive review predicted negative)")

## Error Analysis: Misclassified Reviews

Looking at what the model gets wrong reveals the limits of the architecture. Common failure modes:
- Sarcasm: *"Oh fantastic, another predictable ending"*
- Negation in complex clauses: *"Not unlike the worst films of the decade"*
- Mixed sentiment: positive review that spends most of its text describing flaws before concluding positively

In [ ]:
misclassified_idx = np.where(y_pred_bilstm != y_test)[0]
print(f"Total misclassified: {len(misclassified_idx):,} / {len(y_test):,}")
print(f"Error rate: {len(misclassified_idx) / len(y_test):.1%}")

In [ ]:
# Show 5 misclassified reviews
print("=" * 70)
for idx in misclassified_idx[:5]:
    true_label = "POSITIVE" if y_test[idx] == 1 else "NEGATIVE"
    pred_label = "POSITIVE" if y_pred_bilstm[idx] == 1 else "NEGATIVE"
    prob       = y_proba_bilstm[idx]
    text       = decode_review(X_test_raw[idx], n_words=80)

    print(f"Index: {idx}")
    print(f"  True label  : {true_label}")
    print(f"  Prediction  : {pred_label}  (P(positive) = {prob:.3f})")
    print(f"  Review      : {text}")
    print("-" * 70)

## Predict on New Text

Test the model on reviews you write yourself.

In [ ]:
# Build a word → id mapping for inference on raw text
word_to_id = {k: v + 3 for k, v in imdb.get_word_index().items()}
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2

def encode_review(text, vocab_size=VOCAB_SIZE, max_len=MAX_LEN):
    """Tokenize raw text and encode to a padded integer sequence."""
    tokens = text.lower().split()
    seq = [word_to_id.get(w, 2) for w in tokens   # 2 = <UNK>
           if word_to_id.get(w, 0) < vocab_size]
    return pad_sequences([seq], maxlen=max_len, padding="post", truncating="post")

def predict_sentiment(text, model):
    prob  = model.predict(encode_review(text), verbose=0)[0][0]
    label = "POSITIVE" if prob >= 0.5 else "NEGATIVE"
    return label, float(prob)

In [ ]:
test_reviews = [
    "This film was absolutely brilliant. The performances were outstanding and the direction superb.",
    "Terrible movie. Dull plot, wooden acting, and a complete waste of two hours.",
    "The movie was not bad at all. I actually enjoyed it more than I expected.",
    "Oh great, another sequel nobody asked for. Predictable from start to finish.",
    "Mixed feelings. Some scenes were genuinely moving but the ending ruined everything.",
]

print(f"{'Review':<70}  {'Label':>10}  {'P(pos)':>8}")
print("-" * 92)
for review in test_reviews:
    label, prob = predict_sentiment(review, model_bilstm)
    print(f"{review[:68]:<70}  {label:>10}  {prob:>8.3f}")

## Summary

| Model | Accuracy | AUC | Key advantage |
|-------|----------|-----|---------------|
| BoW + Logistic Regression | ~88% | ~95% | Fast; interpretable (inspect coefficients) |
| LSTM | ~87–88% | ~95% | Handles word order; can learn negation |
| Bidirectional LSTM | ~87–88% | ~95–96% | Reads both directions; slightly more robust |

**Key takeaways:**

1. The BoW baseline is surprisingly strong on IMDB — most sentiment is expressed through individual high-signal words (*terrible*, *brilliant*) that TF-IDF handles well.
2. LSTMs show their advantage on reviews with complex structure — negation, hedging, multi-clause sentences — where word order matters.
3. The gap between models grows on harder datasets (longer texts, more sarcasm, domain-specific language).
4. For production-grade sentiment analysis, pre-trained transformers (DistilBERT, RoBERTa) typically outperform from-scratch LSTMs by a substantial margin without requiring more labeled data.